# Dominion: find actual project assets

This supersedes notebook 05 for **asset coverage**. Its endpoint candidates remain useful research, but an endpoint is not an existing transmission line.

We use the same libraries as before: Python file handling, `json`, `re`, `time`, `requests`, and pandas. `requests.get()` downloads public source responses; pandas keeps project and review rows in tables. Small functions in `gridlock.asset_discovery` handle the steps below. No geometry library is used in this version.

For mixed titles, the **first existing-asset action** (`replace`, `rebuild`, `upgrade`) takes priority over `construct`. All keywords remain in `detected_actions`. Description actions do not change the requested title-based classification.

One project can require several assets. We retain every source object separately, with its original geometry. We never draw a line between endpoints here.

In [1]:
import json
import pandas as pd

from gridlock.paths import DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR
from gridlock.asset_sources import fetch_osm_assets, fetch_public_lines, fetch_official_maps
from gridlock.asset_discovery import (
    prepare_projects, read_asset_requirements, match_basis, search_named_osm_objects,
    find_project_matches, add_official_map_evidence, apply_verified_assets, summarize_coverage,
    print_coverage, save_results,
)

projects_path = PROCESSED_DATA_DIR / "dominion" / "dominion_projects.csv"
projects = pd.read_csv(projects_path, dtype=str, keep_default_na=False)
projects = prepare_projects(projects)

## 1. Read each description as a set of required assets

`extract_identifiers()` collects name clues, voltage, circuit numbers, line identifiers, and phrases following ?at?. These clues are not automatically separate assets: a nearby river, a proposed second circuit, and an existing line mean different things.

For the current 54 SCRTP descriptions, `data/reference/dominion_asset_scope.txt` is a readable, checked inventory of 109 references. Each line gives the source page, asset type/name, endpoints, voltage, circuit/identifier, existing/proposed status, and scope notes. Separate circuits and work segments stay separate. Uncertain scope, such as the Aiken PSA tap, stays unresolved. The complete original description stays in the review table, including numbered structures.

`dominion_scope_source.json` records which exact titles/descriptions were checked. If source text changes, the code requests scope review instead of silently claiming the old list is complete. A new state's data can be downloaded without code changes; its descriptions still need an asset inventory before complete-coverage claims.

Project IDs are not unique in this PDF: `6809 M` occurs on two different pages. `project_key` includes the source file and page to keep both projects.

In [2]:
requirements = read_asset_requirements(projects)

## 2. Bulk OSM infrastructure by project state

`fetch_osm_assets(state)` reuses the saved statewide Overpass responses for substations, lines, plants and switchyards. It keeps tags such as voltage, name and ref. There is no utility/operator filter. Plant boundaries can provide clues but do not count as substation boundaries.

In [3]:
features = []
for state, state_projects in projects.groupby("state"):
    features.extend(fetch_osm_assets(state))

## 3. Nominatim fallback for missing named objects

Search missing facility names or line endpoint pairs programmatically. This is a one-time research pass, with cached responses and at least 1.1 seconds between uncached calls. It does not query every project repeatedly. Keep `run_nominatim = False` to skip this network step; an initial completed run saves all responses for reuse.

Only OSM power objects in the project's state are retained. A town point is not a line or substation. A line result must contain actual line geometry.

[Nominatim usage policy](https://operations.osmfoundation.org/policies/nominatim/)

In [4]:
run_nominatim = True
if run_nominatim:
    for asset in requirements:
        strong_names = ["identifier", "both_endpoint_names", "exact_station_name"]
        found = any(match_basis(asset, feature) in strong_names for feature in features)
        if not found:
            features.extend(search_named_osm_objects(asset))

## 4. Public utility maps, then other public GIS

`fetch_official_maps()` reads Dominion's South Carolina project index in bulk, preserves project pages, and downloads linked map files. Related pages/maps are attached as evidence; a PDF/image without geographic coordinates does not count as a located GIS object.

The public [DOE/NETL transmission layer](https://arcgis.netl.doe.gov/server/rest/services/Hosted/Energy_Transition_Atlas_493d6/FeatureServer/18) supplies source line vertices, endpoint labels and voltage. It is a 2022 HIFLD snapshot retrieved by the publisher in 2023, so newer assets may be missing. The query uses Census bounds for each project state and includes all operators. Bounds are a search rectangle, so bordering-state features may be included as candidates. Each raw response page is saved; unknown endpoints and inferred source attributes remain visible in `source_tags`.

The separately saved Dominion heatmap data is not used for geometry: its published coordinate values are outside latitude/longitude ranges and no validated conversion is available here.

In [5]:
official_maps = fetch_official_maps()
for state, state_projects in projects.groupby("state"):
    features.extend(fetch_public_lines(state))

## 5. Match objects and preserve unresolved scope

`find_project_matches()` checks asset type, state, names/identifiers and voltage. It keeps all candidate source objects, including several objects for one required asset. It uses no score. `source` records Overpass, Nominatim or public GIS; source URLs and raw cache paths remain attached.

- `located`: a uniquely identified existing station with compatible source evidence.
- `candidate`: a real source object exists, but its identity, circuit or required extent still needs verification. **A candidate's geometry is not confirmed project geometry.**
- `unresolved`: no matching source object was found.

For lines, matching endpoint labels does not establish the full required segment. Circuit and scope review are still required before a line counts as located. A parent corridor, plant footprint, nearby station, or proposed route's existing endpoint cannot silently stand in for the requested object. We keep their evidence rather than dropping it.

After checking actual circuit identity and the full required extent against source evidence, `verified_assets` can record a verified source object. Several segments may be verified for one asset only when their combined geometry covers its scope. The function preserves source geometry and requires an evidence note. The dictionary is empty in this run: no line route has been asserted as confirmed.

In [6]:
review = find_project_matches(requirements, features)
review = add_official_map_evidence(review, official_maps)
# Add only reviewed source-object matches with evidence of the full asset scope.
# Keys: (asset_key, feature_id). Values: source/map evidence explaining identity and extent.
verified_assets = {}
review = apply_verified_assets(review, verified_assets)
summary = summarize_coverage(projects, review)
review["project_coverage"] = review["project_key"].map(summary.set_index("project_key")["coverage"])
output_path = save_results(review)

## 6. Coverage of the full referenced asset set

In each `X / Y` line, **X means projects with all referenced assets located**. Mixed construction/rebuild projects need their proposed and existing assets represented before they count as complete. ?Some? means at least one required asset is located; it never means every asset was found.

The only new output table is `data/processed/geo/dominion_asset_review.csv`. It preserves unresolved rows, separate assets/circuits, state, identifiers, original source geometry and evidence. No final project GeoJSON is generated. The earlier endpoint-candidate totals are not comparable to these stricter asset-coverage totals.

In [7]:
print_coverage(summary)

Replace projects: 0 / 5 real assets located
Rebuild projects: 0 / 27 real assets located
Upgrade projects: 0 / 2 real assets located
Construct projects: 0 / 14 locations/routes located
Other projects: 0 / 6 located
Projects with all referenced assets located: 0
Projects with some referenced assets located: 5
Projects with no referenced assets located: 49
